In [1]:
import pandas as pd

df = pd.read_csv('2019-Nov.csv')

df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


In [2]:
smartphone_df = df[df['category_code'] == 'electronics.smartphone'].copy()

In [3]:
#정렬 + unknown처리
df = smartphone_df.sort_values(['user_session', 'event_time']).copy()
df['brand'] = df['brand'].fillna('unknown')

In [4]:
grouped = df.groupby(['user_session', 'product_id'])

In [5]:
event_first_time = (
    smartphone_df
    .groupby(['user_session', 'product_id', 'event_type'])['event_time']
    .min()
    .unstack()
    .reset_index()
)

for col in ['view', 'cart', 'purchase']:
    if col not in event_first_time.columns:
        event_first_time[col] = pd.NaT

# 존재 여부 컬럼 따로 생성
event_first_time['has_view'] = event_first_time['view'].notna()
event_first_time['has_cart'] = event_first_time['cart'].notna()
event_first_time['has_purchase'] = event_first_time['purchase'].notna()

# 순서 검증
event_first_time['view_to_cart'] = (
    event_first_time['has_view'] &
    event_first_time['has_cart'] &
    (event_first_time['view'] < event_first_time['cart'])
)

event_first_time['view_to_cart_to_purchase'] = (
    event_first_time['has_view'] &
    event_first_time['has_cart'] &
    event_first_time['has_purchase'] &
    (event_first_time['view'] < event_first_time['cart']) &
    (event_first_time['cart'] < event_first_time['purchase'])
)

funnel_df = event_first_time[[
    'user_session',
    'product_id',
    'has_view',
    'view_to_cart',
    'view_to_cart_to_purchase'
]].rename(columns={
    'has_view': 'view'
})

In [6]:
funnel_summary = pd.DataFrame({
    'step': ['view', 'view_to_cart', 'view_to_cart_to_purchase'],
    'count': [
        funnel_df['view'].sum(),
        funnel_df['view_to_cart'].sum(),
        funnel_df['view_to_cart_to_purchase'].sum()
    ]
})

funnel_summary['conversion_rate_vs_view'] = (
    funnel_summary['count'] / funnel_summary.loc[0, 'count'] * 100
)

funnel_summary

,step,count,conversion_rate_vs_view
0,view,9566761,100.000000
1,view_to_cart,739062,7.725311
2,view_to_cart_to_purchase,300234,3.138304


In [7]:
view_sessions = funnel_df['view'].sum()
view_to_cart_sessions = funnel_df['view_to_cart'].sum()
view_to_cart_to_purchase_sessions = funnel_df['view_to_cart_to_purchase'].sum()

view_to_cart_rate = view_to_cart_sessions / view_sessions
cart_to_purchase_rate = view_to_cart_to_purchase_sessions / view_to_cart_sessions
total_conversion_rate = view_to_cart_to_purchase_sessions / view_sessions

In [8]:
target_brands = ['samsung', 'apple', 'xiaomi']

df_flow = df[df['brand'].isin(target_brands)].copy()
df_flow = df_flow.sort_values(['user_session', 'event_time'])

flow_df = (
    df_flow[df_flow['event_type'].isin(['view', 'cart', 'purchase'])]
    .groupby(['user_session', 'event_type'])['brand']
    .first()
    .unstack()
    .reset_index()
)

flow_df = flow_df.rename(columns={
    'view': 'view_brand',
    'cart': 'cart_brand',
    'purchase': 'purchase_brand'
})

flow_df.head()

event_type,user_session,cart_brand,purchase_brand,view_brand
0,0000009d-1f5b-40b9-bd23-db4f3d973ae3,NaN,NaN,apple
1,00000616-f016-4c01-b323-438486d9d3ee,NaN,NaN,apple
2,000009c4-a1dd-4764-87d9-24f3d7e43c4f,NaN,NaN,apple
3,00000c2a-aada-4ee8-a0ae-1948795f4d34,NaN,NaN,samsung
4,000015e0-0fe1-440b-9d4b-1395cef84f38,NaN,NaN,apple


In [9]:
def classify_flow(row):
    v = row.get('view_brand')
    c = row.get('cart_brand')
    p = row.get('purchase_brand')

    brands = [x for x in [v, c, p] if pd.notna(x)]

    if pd.isna(v) or pd.isna(p):
        return 'no_purchase_or_no_view'

    if pd.notna(c):
        if v == c == p:
            return 'same_brand'
        elif v != c and c == p:
            return 'view_to_cart_switch'
        elif v == c and c != p:
            return 'cart_to_purchase_switch'
        elif len(set(brands)) == 3:
            return 'multi_brand_exploration'
        else:
            return 'etc'
    else:
        if v == p:
            return 'direct_same_purchase'
        else:
            return 'direct_switch_purchase'

flow_df['pattern'] = flow_df.apply(classify_flow, axis=1)

flow_df['pattern'].value_counts()

pattern
no_purchase_or_no_view     3441903
same_brand                  239880
direct_same_purchase         31412
view_to_cart_switch           9835
direct_switch_purchase        1262
cart_to_purchase_switch       1194
etc                            399
multi_brand_exploration         31
Name: count, dtype: int64

In [10]:
flow_df['pattern'].value_counts(normalize=True) * 100

pattern
no_purchase_or_no_view     92.377364
same_brand                  6.438148
direct_same_purchase        0.843068
view_to_cart_switch         0.263962
direct_switch_purchase      0.033871
cart_to_purchase_switch     0.032046
etc                         0.010709
multi_brand_exploration     0.000832
Name: proportion, dtype: float64

view → cart 변경 분석

In [11]:
vc_switch = flow_df[flow_df['pattern'] == 'view_to_cart_switch'].copy()

vc_matrix = (
    vc_switch.groupby(['view_brand', 'cart_brand'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

vc_matrix

,view_brand,cart_brand,count
3,samsung,xiaomi,2973
0,apple,samsung,2721
2,samsung,apple,2203
5,xiaomi,samsung,1039
1,apple,xiaomi,663
4,xiaomi,apple,236


cart → purchase 변경 분석

In [12]:
cp_switch = flow_df[flow_df['pattern'] == 'cart_to_purchase_switch'].copy()

cp_matrix = (
    cp_switch.groupby(['cart_brand', 'purchase_brand'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

cp_matrix

,cart_brand,purchase_brand,count
0,apple,samsung,381
3,samsung,xiaomi,339
2,samsung,apple,260
5,xiaomi,samsung,139
1,apple,xiaomi,52
4,xiaomi,apple,23


direct 구매 변경 (view → purchase)

In [13]:
dp_switch = flow_df[flow_df['pattern'] == 'direct_switch_purchase'].copy()

dp_matrix = (
    dp_switch.groupby(['view_brand', 'purchase_brand'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

dp_matrix

,view_brand,purchase_brand,count
3,samsung,xiaomi,382
2,samsung,apple,319
0,apple,samsung,308
5,xiaomi,samsung,128
1,apple,xiaomi,97
4,xiaomi,apple,28


In [14]:
vc_ratio = vc_matrix.copy()

vc_ratio['total_from_view'] = vc_ratio.groupby('view_brand')['count'].transform('sum')
vc_ratio['ratio'] = vc_ratio['count'] / vc_ratio['total_from_view'] * 100

vc_ratio.sort_values(['view_brand', 'ratio'], ascending=[True, False])

,view_brand,cart_brand,count,total_from_view,ratio
0,apple,samsung,2721,3384,80.407801
1,apple,xiaomi,663,3384,19.592199
3,samsung,xiaomi,2973,5176,57.438176
2,samsung,apple,2203,5176,42.561824
5,xiaomi,samsung,1039,1275,81.490196
4,xiaomi,apple,236,1275,18.509804


In [15]:
cp_ratio = cp_matrix.copy()

cp_ratio['total_from_cart'] = cp_ratio.groupby('cart_brand')['count'].transform('sum')
cp_ratio['ratio'] = cp_ratio['count'] / cp_ratio['total_from_cart'] * 100

cp_ratio.sort_values(['cart_brand', 'ratio'], ascending=[True, False])

,cart_brand,purchase_brand,count,total_from_cart,ratio
0,apple,samsung,381,433,87.990762
1,apple,xiaomi,52,433,12.009238
3,samsung,xiaomi,339,599,56.594324
2,samsung,apple,260,599,43.405676
5,xiaomi,samsung,139,162,85.802469
4,xiaomi,apple,23,162,14.197531


In [16]:
dp_ratio = dp_matrix.copy()

dp_ratio['total_from_view'] = dp_ratio.groupby('view_brand')['count'].transform('sum')
dp_ratio['ratio'] = dp_ratio['count'] / dp_ratio['total_from_view'] * 100

dp_ratio.sort_values(['view_brand', 'ratio'], ascending=[True, False])

,view_brand,purchase_brand,count,total_from_view,ratio
0,apple,samsung,308,405,76.049383
1,apple,xiaomi,97,405,23.950617
3,samsung,xiaomi,382,701,54.493581
2,samsung,apple,319,701,45.506419
5,xiaomi,samsung,128,156,82.051282
4,xiaomi,apple,28,156,17.948718


구매한 브랜드 기준 소요시간(브랜드 관여)


같은 브랜드를 보고 선택한 소요시간(동일 브랜드)

In [ ]:
# view / purchase 시간 + 브랜드 추출
event_time_df = (
    df[df['event_type'].isin(['view', 'purchase'])]
    .sort_values(['user_session', 'event_time'])
)

event_time_pivot = (
    event_time_df
    .groupby(['user_session', 'event_type'])
    .agg({
        'event_time': 'first',
        'brand': 'first'
    })
    .unstack()
)

# 컬럼 정리
event_time_pivot.columns = [
    f"{col[0]}_{col[1]}" for col in event_time_pivot.columns
]

event_time_pivot = event_time_pivot.reset_index()

event_time_pivot.head()

In [ ]:
# event_time datetime 변환
df['event_time'] = pd.to_datetime(df['event_time'], errors='coerce')

# 다시 pivot 생성
event_time_df = (
    df[df['event_type'].isin(['view', 'purchase'])]
    .sort_values(['user_session', 'event_time'])
)

event_time_pivot = (
    event_time_df
    .groupby(['user_session', 'event_type'])
    .agg({
        'event_time': 'first',
        'brand': 'first'
    })
    .unstack()
)

event_time_pivot.columns = [
    f"{col[0]}_{col[1]}" for col in event_time_pivot.columns
]

event_time_pivot = event_time_pivot.reset_index()

# 혹시 모를 문자열 방지
event_time_pivot['event_time_view'] = pd.to_datetime(
    event_time_pivot['event_time_view'], errors='coerce'
)

event_time_pivot['event_time_purchase'] = pd.to_datetime(
    event_time_pivot['event_time_purchase'], errors='coerce'
)

# 구매 도달 시간 계산
event_time_pivot['time_to_purchase_sec'] = (
    event_time_pivot['event_time_purchase'] - event_time_pivot['event_time_view']
).dt.total_seconds()

event_time_pivot['time_to_purchase_hour'] = (
    event_time_pivot['time_to_purchase_sec'] / 3600
)

event_time_pivot.head()

In [ ]:
purchase_time_df = event_time_pivot[
    event_time_pivot['event_time_view'].notna() &
    event_time_pivot['event_time_purchase'].notna()
].copy()

In [ ]:
purchase_time_by_brand = (
    purchase_time_df.groupby('brand_purchase')
    .agg(
        avg_time=('time_to_purchase_hour', 'mean'),
        median_time=('time_to_purchase_hour', 'median'),
        count=('time_to_purchase_hour', 'count')
    )
    .reset_index()
    .sort_values('avg_time')
)

purchase_time_by_brand

In [ ]:
same_brand_df = purchase_time_df[
    purchase_time_df['brand_view'] == purchase_time_df['brand_purchase']
].copy()

same_brand_time = (
    same_brand_df.groupby('brand_purchase')
    .agg(
        avg_time=('time_to_purchase_hour', 'mean'),
        median_time=('time_to_purchase_hour', 'median'),
        count=('time_to_purchase_hour', 'count')
    )
    .reset_index()
    .sort_values('avg_time')
)

same_brand_time

In [ ]:
purchase_time_df['same_brand'] = (
    purchase_time_df['brand_view'] == purchase_time_df['brand_purchase']
)

compare_time = (
    purchase_time_df.groupby(['brand_purchase', 'same_brand'])
    .agg(
        avg_time=('time_to_purchase_hour', 'mean'),
        median_time=('time_to_purchase_hour', 'median'),
        count=('time_to_purchase_hour', 'count')
    )
    .reset_index()
)

compare_time

In [ ]:
target_brands = ['samsung', 'apple', 'xiaomi']

purchase_time_by_brand = purchase_time_by_brand[
    purchase_time_by_brand['brand_purchase'].isin(target_brands)
]

same_brand_time = same_brand_time[
    same_brand_time['brand_purchase'].isin(target_brands)
]

compare_time = compare_time[
    compare_time['brand_purchase'].isin(target_brands)
]

In [ ]:
# 전체 vs 동일 브랜드 비교
final_time_compare = purchase_time_by_brand.merge(
    same_brand_time,
    on='brand_purchase',
    suffixes=('_all', '_same')
)

# 보기 좋게 정리
final_time_compare = final_time_compare[
    [
        'brand_purchase',
        'avg_time_all', 'median_time_all',
        'avg_time_same', 'median_time_same',
        'count_all', 'count_same'
    ]
]

# 차이 계산
final_time_compare['time_diff'] = (
    final_time_compare['avg_time_all'] - final_time_compare['avg_time_same']
)

final_time_compare.sort_values('avg_time_all')

In [ ]:
# same_brand True/False를 보기 좋게
compare_time_pivot = compare_time.pivot(
    index='brand_purchase',
    columns='same_brand',
    values='avg_time'
).reset_index()

compare_time_pivot.columns = [
    'brand',
    'cross_purchase_time',  # False
    'same_brand_time'       # True
]

# 차이 계산
compare_time_pivot['diff'] = (
    compare_time_pivot['cross_purchase_time'] -
    compare_time_pivot['same_brand_time']
)

compare_time_pivot.sort_values('cross_purchase_time')

In [ ]:
import matplotlib.pyplot as plt

x = range(len(compare_time_pivot))

plt.figure(figsize=(10,6))

plt.bar(
    [i - 0.2 for i in x],
    compare_time_pivot['same_brand_time'],
    width=0.4,
    label='Same Brand'
)

plt.bar(
    [i + 0.2 for i in x],
    compare_time_pivot['cross_purchase_time'],
    width=0.4,
    label='Cross Brand'
)

plt.xticks(x, compare_time_pivot['brand'])
plt.ylabel('Time to Purchase (hours)')
plt.title('Same vs Cross Brand Purchase Time')
plt.legend()

plt.show()

In [ ]:
# 시간 구간화
purchase_time_df['time_bucket'] = pd.cut(
    purchase_time_df['time_to_purchase_hour'],
    bins=[0, 1, 6, 24, 48, 999],
    labels=['<1h', '1-6h', '6-24h', '1-2d', '2d+']
)

# 브랜드별 분포
time_dist = (
    purchase_time_df.groupby(['brand_purchase', 'time_bucket'])
    .size()
    .reset_index(name='count')
)

time_dist

대분류 컬럼 생성

In [ ]:
df['main_category'] = df['category_code'].str.split('.').str[0]

In [ ]:
df['main_category'].value_counts()

In [ ]:
event_first_time_main = (
    df
    .groupby([
        'main_category',
        'user_session',
        'product_id',
        'event_type'
    ])['event_time']
    .min()
    .unstack()
    .reset_index()
)

In [ ]:
for col in ['view', 'cart', 'purchase']:
    if col not in event_first_time_main.columns:
        event_first_time_main[col] = pd.NaT

# 존재 여부
event_first_time_main['has_view'] = event_first_time_main['view'].notna()
event_first_time_main['has_cart'] = event_first_time_main['cart'].notna()
event_first_time_main['has_purchase'] = event_first_time_main['purchase'].notna()

# 순차 퍼널
event_first_time_main['view_to_cart'] = (
    event_first_time_main['has_view'] &
    event_first_time_main['has_cart'] &
    (event_first_time_main['view'] < event_first_time_main['cart'])
)

event_first_time_main['view_to_cart_to_purchase'] = (
    event_first_time_main['has_view'] &
    event_first_time_main['has_cart'] &
    event_first_time_main['has_purchase'] &
    (event_first_time_main['view'] < event_first_time_main['cart']) &
    (event_first_time_main['cart'] < event_first_time_main['purchase'])
)

In [ ]:
funnel_df = event_first_time_main[[
    'main_category',
    'user_session',
    'product_id',
    'has_view',
    'view_to_cart',
    'view_to_cart_to_purchase'
]].rename(columns={
    'has_view': 'view'
})

In [ ]:
category_funnel = (
    funnel_df.groupby('main_category')
    .agg(
        view=('view', 'sum'),
        view_to_cart=('view_to_cart', 'sum'),
        conversion=('view_to_cart_to_purchase', 'sum')
    )
    .reset_index()
)

In [ ]:
category_funnel['view_to_cart_rate'] = (
    category_funnel['view_to_cart'] /
    category_funnel['view'] * 100
)

category_funnel['cart_to_purchase_rate'] = (
    category_funnel['conversion'] /
    category_funnel['view_to_cart'] * 100
)

category_funnel['total_conversion_rate'] = (
    category_funnel['conversion'] /
    category_funnel['view'] * 100
)

In [ ]:
category_funnel = category_funnel.sort_values(
    'total_conversion_rate',
    ascending=False
)

category_funnel

소분류 컬럼 생성 sub_category

In [ ]:
event_first_time_sub = (
    df
    .groupby([
        'sub_category',
        'user_session',
        'product_id',
        'event_type'
    ])['event_time']
    .min()
    .unstack()
    .reset_index()
)

In [ ]:
for col in ['view', 'cart', 'purchase']:
    if col not in event_first_time_sub.columns:
        event_first_time_sub[col] = pd.NaT

# 존재 여부
event_first_time_sub['has_view'] = (
    event_first_time_sub['view'].notna()
)

event_first_time_sub['has_cart'] = (
    event_first_time_sub['cart'].notna()
)

event_first_time_sub['has_purchase'] = (
    event_first_time_sub['purchase'].notna()
)

# 순차 퍼널
event_first_time_sub['view_to_cart'] = (
    event_first_time_sub['has_view'] &
    event_first_time_sub['has_cart'] &
    (event_first_time_sub['view'] < event_first_time_sub['cart'])
)

event_first_time_sub['view_to_cart_to_purchase'] = (
    event_first_time_sub['has_view'] &
    event_first_time_sub['has_cart'] &
    event_first_time_sub['has_purchase'] &
    (event_first_time_sub['view'] < event_first_time_sub['cart']) &
    (event_first_time_sub['cart'] < event_first_time_sub['purchase'])
)

In [ ]:
funnel_df_sub = event_first_time_sub[[
    'sub_category',
    'user_session',
    'product_id',
    'has_view',
    'view_to_cart',
    'view_to_cart_to_purchase'
]].rename(columns={
    'has_view': 'view'
})

In [ ]:
sub_category_funnel_summary = (
    funnel_df_sub.groupby('sub_category')
    .agg(
        view=('view', 'sum'),
        view_to_cart=('view_to_cart', 'sum'),
        view_to_cart_to_purchase=('view_to_cart_to_purchase', 'sum')
    )
    .reset_index()
)

In [ ]:
sub_category_funnel_summary['view_to_cart_rate'] = (
    sub_category_funnel_summary['view_to_cart'] /
    sub_category_funnel_summary['view'] * 100
)

sub_category_funnel_summary['cart_to_purchase_rate'] = (
    sub_category_funnel_summary['view_to_cart_to_purchase'] /
    sub_category_funnel_summary['view_to_cart'] * 100
)

sub_category_funnel_summary['total_conversion_rate'] = (
    sub_category_funnel_summary['view_to_cart_to_purchase'] /
    sub_category_funnel_summary['view'] * 100
)

In [ ]:
sub_category_funnel_summary = (
    sub_category_funnel_summary
    .sort_values(
        'total_conversion_rate',
        ascending=False
    )
)

sub_category_funnel_summary